In [76]:
"""
repo_scaffold.py

Scaffolds a generalized ingestion-pipeline repository structure.
Usage:
  python repo_scaffold.py \
    --repo-root client-datalake-pipelines \
    --client myclient --env dev --domain sap \
    --entity orders --source appflow --target s3 --action ingest
"""
import argparse
import sys
from pathlib import Path
from textwrap import dedent

# Templates for generated files
TEMPLATE_README = '''# {repo_name}

This repository contains ingestion pipelines for {client}.

Structure:
- common/: shared modules
- libs/: zipped shared code
- scripts/: pipeline scripts for Glue and other services
- workflows/: Glue Workflow definitions
- infrastructure/: IaC placeholders
- tests/: test stubs
'''

SCRIPT_TEMPLATE = '''"""
{script_filename}

Glue ingestion script placeholder for:
client={client}, domain={domain}, entity={entity}, source={source}, target={target}, action={action}
"""
from common.config import parse_args
from common.logger import get_logger
from common.utils import build_s3_ingest_path

def run():
    args = parse_args()
    # TODO: implement pipeline logic

if __name__ == "__main__":
    run()
'''

TEST_TEMPLATE = '''"""
Test stub for {domain}-{entity} pipeline
"""
from common.utils import build_s3_ingest_path

def test_build_path():
    path = build_s3_ingest_path("{domain}", "{entity}", "2025-01-01")
    assert path.startswith("s3://")
'''

WORKFLOW_TEMPLATE = '''# {domain}_workflow.py

# Glue workflow definition placeholder for {domain}
'''
# Clear sys.argv except for the first element (script name)
sys.argv = ['']  # resetting args so parser doesn't crash in notebook

def create_repo_structure(args):
    client, env, domain, entity, source, target, action, connect = args.client, args.env, args.domain, args.entity, args.source, args.target, args.action, args.connect
    
    root = Path(f'{client}-datalake-pipelines')
    if root.exists():
        sys.exit(f"Error: Repository root '{root}' already exists. Aborting.")
    root.mkdir(parents=True)

    # Root README
    (root / 'README.md').write_text(
        TEMPLATE_README.format(repo_name=root.name, client=client)
    )

    # common/
    common = root / 'common'
    common.mkdir()
    for fname in ('__init__.py', 'config.py', 'logger.py', 'utils.py'):
        (common / fname).touch()

    # libs/
    libs = root / 'libs'
    libs.mkdir()
    # Create placeholder zip file
    zipname = f"{client}-common.zip"
    (libs / zipname).touch()

    # scripts/glue/... pipeline folder
    scripts = root / 'scripts' / 'glue'
    scripts.mkdir(parents=True)
    folder_name = f"{client}-{domain}-{entity}-{source}-{connect}-{target}-{action}"
    pipeline_dir = scripts / folder_name
    pipeline_dir.mkdir()

    script_filename = f"{client}_{folder_name}.py"
    script_file = pipeline_dir / script_filename
    script_file.write_text(
        SCRIPT_TEMPLATE.format(
            script_filename=script_filename,
            client=client,
            domain=domain,
            entity=entity,
            source=source,
            target=target,
            action=action
        )
    )
    # requirements.txt
    (pipeline_dir / 'requirements.txt').touch()

    # workflows/
    workflows = root / 'workflows'
    workflows.mkdir()
    wf_file = workflows / f"{domain}_workflow.py"
    wf_file.write_text(WORKFLOW_TEMPLATE.format(domain=domain))

    # infrastructure/
    infra = root / 'infrastructure'
    infra.mkdir()
    (infra / 'README.md').write_text("# Infrastructure-as-Code placeholders\n")

    # tests/
    tests = root / 'tests'
    tests.mkdir()
    test_file = tests / f"test_{domain}_{entity}.py"
    test_file.write_text(
        TEST_TEMPLATE.format(domain=domain, entity=entity)
    )

    print(f"Scaffolded repository at {root.resolve()}")


# S3 Bucket Naming Strategy

## Overview

The S3 bucket naming strategy follows the pattern: `{client}-{env}-{zone}`

- `{client}`: Client's name or identifier (e.g., "myclient")
- `{env}`: Environment (e.g., "dev", "staging", "prod")
- `{zone}`: One of the following zones:

| Zone | Purpose |
| --- | --- |
| ingest | Raw incoming data |
| structured | Cleaned and transformed datasets |
| scripts | Glue ETL scripts |
| temp | Temporary or intermediate data |
| archive | Historical backups |
| metadata | Operational metadata, logs |

For example, for client "myclient" in the "dev" environment, the buckets would be:

- `myclient-dev-ingest`
- `myclient-dev-structured`
- `myclient-dev-scripts`
- `myclient-dev-temp`
- `myclient-dev-archive`
- `myclient-dev-metadata`

## Folder Structures

### Ingest Zone

The ingest zone stores raw incoming data, organized by source and entity, with optional partitioning by filter values.

```plaintext
s3://{client}-{env}-ingest/
└── {source}/
    ├── {entity}/
    │   └── filter=filter_value/
    │       └── {entity}_YYYYMMDDHHMMSS.csv
    └── {entity}/
        ├── {entity}_YYYYMMDDHHMMSS.csv
        └── {entity}_YYYYMMDDHHMMSS.csv
```

### Structured Zone

The structured zone contains cleaned and transformed datasets, following a similar structure to the ingest zone.

```plaintext
s3://{client}-{env}-structured/
└── {source}/
    ├── {entity}/
    │   └── filter=filter_value/
    │       └── {entity}_YYYYMMDDHHMMSS.csv
    └── {entity}/
        ├── {entity}_YYYYMMDDHHMMSS.csv
        └── {entity}_YYYYMMDDHHMMSS.csv
```

### Scripts Zone

The scripts zone holds service-specific artifacts such as Glue job scripts, Athena query definitions, and Lambda functions.

```plaintext
s3://{client}-{env}-scripts/
├── glue/
│   ├── {domain}_{entity}_{source}_{connect}_{target}_{action}/
│   │   ├── {client}_{domain}_{entity}_{source}_{connect}_{target}_{action}.py
│   │   └── requirements.txt
│   └── ...
├── athena/
│   ├── {entity}_query.sql
│   └── {another_entity}_ltv_view.sql
└── lambda/
    ├── {function_name}/
    │   ├── handler.py
    │   └── requirements.txt
```

### Temp Zone

The temp zone is used for temporary or intermediate data, including Athena query results and other intermediate files.

```plaintext
s3://{client}-{env}-temp/
├── athena-query-results/
└── intermediate/
```

### Archive Zone

The archive zone stores historical backups, organized similarly to the ingest and structured zones.

```plaintext
s3://{client}-{env}-archive/
└── {source}/
    ├── {entity}/
    │   └── filter=filter_value/
    │       └── {entity}_YYYYMMDDHHMMSS.csv
    └── {entity}/
        ├── {entity}_YYYYMMDDHHMMSS.csv
        └── {entity}_YYYYMMDDHHMMSS.csv
```

### Metadata Zone

The metadata zone contains operational metadata, including run logs, configurations, schemas, and lineage information.

```plaintext
s3://{client}-{env}-metadata/
├── runs/
│   └── {source}/{entity}/
│       └── run_date=YYYY-MM-DD/
│           ├── summary-{runId}.json
│           └── details-{runId}.json
├── configs/
│   ├── source_configs.json
│   └── workflow_configs.json
├── schemas/
│   └── {source}-{entity}-schema.json
└── lineage/
    └── {source}-{entity}-lineage.json
```

In [ ]:
import argparse, sys
from datetime import datetime
# Clear sys.argv except for the first element (script name)
sys.argv = ['']  # resetting args so parser doesn't crash in notebook

def generate_s3_names(args):
    client, env, domain, entity, source, target, action, connect = args.client, args.env, args.domain, args.entity, args.source, args.target, args.action, args.connect
    # Define the S3 zones
    zones = ["ingest", "structured", "scripts", "temp", "archive", "metadata"]
    
    # Base bucket names
    buckets = {zone: f"{client}-{env}-{zone}" for zone in zones}
    
    # Current timestamp for file examples
    ts = datetime.now().strftime("%Y%m%d%H%M%S")
    date_iso = datetime.now().strftime("%Y-%m-%d")
    
    # Example filter placeholder
    filter_field = "filter"
    filter_value = "filter_value"
    
    # Build example prefixes per zone
    prefixes = {
        "ingest": [
            f"s3://{buckets['ingest']}/{source}/{entity}/",
            f"s3://{buckets['ingest']}/{source}/{entity}/{filter_field}={filter_value}/{entity}_{ts}.csv"
        ],
        "structured": [
            f"s3://{buckets['structured']}/{source}/{entity}/",
            f"s3://{buckets['structured']}/{source}/{entity}/{filter_field}={filter_value}/{entity}_{ts}.csv"
        ],
        "scripts": [
            f"s3://{buckets['scripts']}/glue/{client}-{domain}-{entity}-{source}-{connect}-{target}-{action}/",
            f"s3://{buckets['scripts']}/glue/{domain}_{entity}_{source}_{connect}_{target}_{action}/{client}-{domain}-{entity}-{source}-{connect}-{target}-{action}.py"
        ],
        "temp": [
            f"s3://{buckets['temp']}/athena-query-results/",
            f"s3://{buckets['temp']}/intermediate/"
        ],
        "archive": [
            f"s3://{buckets['archive']}/{source}/{entity}/{ts}/"
        ],
        "metadata": [
            f"s3://{buckets['metadata']}/runs/{source}/{entity}/run_date={date_iso}/summary-RUNID.json",
            f"s3://{buckets['metadata']}/runs/{source}/{entity}/run_date={date_iso}/details-RUNID.json",
            f"s3://{buckets['metadata']}/configs/source_configs.json",
            f"s3://{buckets['metadata']}/schemas/{source}-{entity}-schema.json",
            f"s3://{buckets['metadata']}/lineage/{source}-{entity}-lineage.json"
        ]
    }
    
    print("\nS3 Bucket Names:")
    for zone, name in buckets.items():
        print(f"  {zone}: {name}")
    
    print("\nExample S3 Paths:")
    for zone, paths in prefixes.items():
        print(f"\n{zone.capitalize()} Zone:")
        for p in paths:
            print(f"  {p}")

In [69]:
import argparse, sys
# Clear sys.argv except for the first element (script name)
sys.argv = ['']  # resetting args so parser doesn't crash in notebook

def generate_glue_names(args):
    client, env, domain, entity, source, target, action, connect = args.client, args.env, args.domain, args.entity, args.source, args.target, args.action, args.connect
    """
    Generate AWS Glue naming conventions including job, workflow, crawler,
    catalog databases, tables, job folder, and script file name.
    """
    # Glue service names
    glue_job = f"gluejob_{env}_{client}_{domain}_{entity}_{source}_{connect}_{target}_{action}"
    glue_workflow = f"gluewf_{env}_{client}_{domain}_{entity}_{source}_{connect}_{target}_{action}"
    glue_crawler = f"gluecr_{env}_{client}_{domain}_{entity}_{source}_{connect}_{target}_{action}"

    # Glue job folder and script names
    glue_job_folder = f"{client}-{domain}-{entity}-{source}-{connect}-{target}-{action}"
    script_name = f"{client}_{domain}_{entity}_{source}_{connect}_{target}_{action}.py"

    # Catalog databases
    catalog_db_ingest = f"{client}_{env}_ingest"
    catalog_db_structured = f"{client}_{env}_structured"
    catalog_db_metadata = f"{client}_{env}_metadata"

    # Catalog table
    catalog_table = f"{domain}_{entity}"

    print("glue_job:", glue_job)
    print("glue_workflow:", glue_workflow)
    print("glue_crawler:", glue_crawler)
    print("glue_job_folder:", glue_job_folder)
    print("script_name:", script_name)
    print("catalog_db_ingest:", catalog_db_ingest)
    print("catalog_db_structured:", catalog_db_structured)
    print("catalog_db_metadata:", catalog_db_metadata)
    print("catalog_table:", catalog_table)

def initialize_names(cmd_args):
    parser = argparse.ArgumentParser(description="Generate AWS Glue service names based on conventions.")
    parser.add_argument("--client", type=str,
                        default="myclient",
                        help="Client name")
    parser.add_argument("--env", type=str,
                        default="dev",
                        choices=["dev", "qa", "prod", "preprod", "staging"],
                        help="Environment")
    parser.add_argument("--domain", type=str,
                        default="services",
                        choices=["services", "database"],
                        help="Domain area")
    parser.add_argument("--entity", type=str,
                        default="tbl",
                        help="Entity/table name")
    parser.add_argument("--source", type=str,
                        default="sapappflow",
                        help="Data source")
    parser.add_argument("--target", type=str,
                        default="s3",
                        choices=["s3", "redshift"],
                        help="Target destination")
    parser.add_argument("--action", type=str,
                        default="ingest",
                        choices=["ingesthistory", "ingestcdc"],
                        help="Action type")
    parser.add_argument("--connect", type=str,
                        default="appflow",
                        help="Connection type")

    args = parser.parse_args(cmd_args)
    return args

In [ ]:
runtime_args = [
    "--client", "myclient",
    "--env",    "prod",
    "--domain", "services",
    "--entity", "enty",
    "--source", "sap",
    "--target", "s3",
    "--action", "ingestcdc",
    "--connect", "appflow"
    ]

args = initialize_names(runtime_args)
generate_glue_names(args)
generate_s3_names(args)

create_repo_structure(args)

glue_job: gluejob_prod_myclient_services_enty_sap_appflow_s3_ingestcdc
glue_workflow: gluewf_prod_myclient_services_enty_sap_appflow_s3_ingestcdc
glue_crawler: gluecr_prod_myclient_services_enty_sap_appflow_s3_ingestcdc
glue_job_folder: myclient-services-enty-sap-appflow-s3-ingestcdc
script_name: myclient_services_enty_sap_appflow_s3_ingestcdc.py
catalog_db_ingest: myclient_prod_ingest
catalog_db_structured: myclient_prod_structured
catalog_db_metadata: myclient_prod_metadata
catalog_table: services_enty

S3 Bucket Names:
  ingest: myclient-prod-ingest
  structured: myclient-prod-structured
  scripts: myclient-prod-scripts
  temp: myclient-prod-temp
  archive: myclient-prod-archive
  metadata: myclient-prod-metadata

Example S3 Paths:

Ingest Zone:
  s3://myclient-prod-ingest/sap/enty/
  s3://myclient-prod-ingest/sap/enty/filter=filter_value/enty_20250506140136.csv

Structured Zone:
  s3://myclient-prod-structured/sap/enty/
  s3://myclient-prod-structured/sap/enty/filter=filter_value/e

In [56]:
a, b = [1, 2]

In [57]:
b

2